# IOI circuit in GPT-2

Trying to reproduce the name-mover heads from the interpretability in the wild paper.

In [ ]:
# only needed on a fresh Colab / clean env
try:
    import transformer_lens
except ImportError:
    %pip install -q transformer_lens

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 136.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 117.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━

In [ ]:
from functools import partial

import einops
import numpy as np
import plotly.express as px
import torch

!pip install -q transformer_lens
!pip uninstall -q -y torchaudio torchvision
from transformer_lens import HookedTransformer, utils
torch.set_grad_enabled(False)

/tmp/ipykernel_9722/2281696659.py:10: DeprecationWarning: The 'utils' module has been deprecated. Please use 'transformer_lens.utilities' instead. Importing from utils.py will be removed in TransformerLens 4.0.
  from transformer_lens import HookedTransformer, utils


In [ ]:
def imshow(tensor, **kwargs):
    px.imshow(
        utils.to_numpy(tensor),
        color_continuous_midpoint=0.0,
        color_continuous_scale="RdBu",
        **kwargs,
    ).show()

Load GPT-2 small.

In [ ]:
device = utils.get_device()
model = HookedTransformer.from_pretrained("gpt2", device=device)
print(device, model.cfg.n_layers, "layers", model.cfg.n_heads, "heads")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Loaded pretrained model gpt2 into HookedTransformer
cuda 12 layers 12 heads


In [ ]:
utils.test_prompt(
    "When John and Mary went to the shops, John gave the bag to",
    " Mary",
    model,
    prepend_bos=True,
)

Tokenized prompt: ['<|endoftext|>', 'When', ' John', ' and', ' Mary', ' went', ' to', ' the', ' shops', ',', ' John', ' gave', ' the', ' bag', ' to']
Tokenized answer: [' Mary']


Performance on answer token:
Rank: 0        Logit: 18.19 Prob: 69.93% Token: | Mary|

Top 0th token. Logit: 18.19 Prob: 69.93% Token: | Mary|
Top 1th token. Logit: 15.82 Prob:  6.49% Token: | them|
Top 2th token. Logit: 15.48 Prob:  4.66% Token: | the|
Top 3th token. Logit: 14.93 Prob:  2.66% Token: | his|
Top 4th token. Logit: 14.86 Prob:  2.49% Token: | John|
Top 5th token. Logit: 14.12 Prob:  1.19% Token: | her|
Top 6th token. Logit: 13.99 Prob:  1.04% Token: | their|
Top 7th token. Logit: 13.70 Prob:  0.78% Token: | a|
Top 8th token. Logit: 13.53 Prob:  0.66% Token: | him|
Top 9th token. Logit: 13.39 Prob:  0.57% Token: | Mrs|


Ranks of the answer tokens: [(' Mary', 0)]

## Prompts

Five templates, each run twice with the two names swapped so the correct answer flips.



In [ ]:
prompt_format = [
    "When John and Mary went to the shops,{} gave the bag to",
    "When Tom and James went to the park,{} gave the ball to",
    "When Dan and Sid went to the shops,{} gave an apple to",
    "After Martin and Amy went to the park,{} gave a drink to",
    "When John and Mary went to the park,{} gave the ball to",
]
# (indirect object = correct, subject = incorrect) for each template
names = [
    (" Mary", " John"),
    (" Tom", " James"),
    (" Dan", " Sid"),
    (" Martin", " Amy"),
    (" Mary", " John"),
]

prompts, answers, answer_tokens = [], [], []
for i in range(len(prompt_format)):
    for j in range(2):
        correct, incorrect = names[i][j], names[i][1 - j]
        answers.append((correct, incorrect))
        answer_tokens.append(
            (model.to_single_token(correct), model.to_single_token(incorrect))
        )
        # subject = the incorrect name, so the correct one is the indirect object
        prompts.append(prompt_format[i].format(incorrect))
answer_tokens = torch.tensor(answer_tokens).to(device)

# all prompts must tokenize to the same length
lengths = {len(model.to_str_tokens(p)) for p in prompts}
assert len(lengths) == 1, f"prompt lengths differ: {lengths} (swap names so they match)"
print(len(prompts), "prompts, length", lengths.pop(), "tokens each")
for p in prompts:
    print(p)

10 prompts, length 15 tokens each
When John and Mary went to the shops, John gave the bag to
When John and Mary went to the shops, Mary gave the bag to
When Tom and James went to the park, James gave the ball to
When Tom and James went to the park, Tom gave the ball to
When Dan and Sid went to the shops, Sid gave an apple to
When Dan and Sid went to the shops, Dan gave an apple to
After Martin and Amy went to the park, Amy gave a drink to
After Martin and Amy went to the park, Martin gave a drink to
When John and Mary went to the park, John gave the ball to
When John and Mary went to the park, Mary gave the ball to


## Clean run + the metric

Metric is the logit difference: `logit(correct) - logit(incorrect)` at the final position,
averaged over prompts.

In [ ]:
def logits_to_ave_logit_diff(logits, answer_tokens, per_prompt=False):
    final_logits = logits[:, -1, :]
    answer_logits = final_logits.gather(dim=-1, index=answer_tokens)
    diff = answer_logits[:, 0] - answer_logits[:, 1]
    return diff if per_prompt else diff.mean()


tokens = model.to_tokens(prompts, prepend_bos=True)
clean_logits, clean_cache = model.run_with_cache(tokens)
clean_logit_diff = logits_to_ave_logit_diff(clean_logits, answer_tokens)
print("clean logit diff:", round(clean_logit_diff.item(), 3))

clean logit diff: 3.484


## Corrupted run



In [ ]:
corrupted_prompts = []
for i in range(0, len(prompts), 2):
    corrupted_prompts.append(prompts[i + 1])
    corrupted_prompts.append(prompts[i])
corrupted_tokens = model.to_tokens(corrupted_prompts, prepend_bos=True)

corrupted_logits = model(corrupted_tokens)
corrupted_logit_diff = logits_to_ave_logit_diff(corrupted_logits, answer_tokens)
print("corrupted logit diff:", round(corrupted_logit_diff.item(), 3))

corrupted logit diff: -3.484


## Patching loop

Run the model on the corrupted tokens, one head at a time, overwrite that head's
output `z`  with the value from the clean cache,
across all positions. then measure logit diff.



In [ ]:
def patch_head_z(corrupted_z, hook, head, clean_cache):
    corrupted_z = corrupted_z.clone()
    corrupted_z[:, :, head, :] = clean_cache[hook.name][:, :, head, :]
    return corrupted_z


def normalize(patched_logit_diff):
    return (patched_logit_diff - corrupted_logit_diff) / (
        clean_logit_diff - corrupted_logit_diff
    )


n_layers, n_heads = model.cfg.n_layers, model.cfg.n_heads
recovered = torch.zeros(n_layers, n_heads, device=device)

for layer in range(n_layers):
    for head in range(n_heads):
        hook_fn = partial(patch_head_z, head=head, clean_cache=clean_cache)
        patched_logits = model.run_with_hooks(
            corrupted_tokens,
            fwd_hooks=[(utils.get_act_name("z", layer, "attn"), hook_fn)],
            return_type="logits",
        )
        ld = logits_to_ave_logit_diff(patched_logits, answer_tokens)
        recovered[layer, head] = normalize(ld)
    print(f"layer {layer} done")

layer 0 done
layer 1 done
layer 2 done
layer 3 done
layer 4 done
layer 5 done
layer 6 done
layer 7 done
layer 8 done
layer 9 done
layer 10 done
layer 11 done


## Heatmap

In [ ]:
imshow(
    recovered,
    labels={"x": "Head", "y": "Layer"},
    title="Logit diff recovered by patching each head's output",
)

# print the top few so the names are easy to read off
vals, idx = recovered.flatten().abs().topk(8)
print("strongest heads (|recovered|):")
for v, i in zip(vals.tolist(), idx.tolist()):
    l, h = divmod(i, n_heads)
    print(f"  L{l}H{h}: {recovered[l, h].item():+.3f}")

strongest heads (|recovered|):
  L10H7: -0.469
  L5H5: +0.321
  L8H6: +0.297
  L8H10: +0.277
  L11H10: -0.254
  L9H9: +0.247
  L7H9: +0.189
  L7H3: +0.131


## What I found
The heatmap effectively maps the broad shape of the IOI circuit, highlighting L10H7 (-0.47) and L11H10 (-0.25) as negative name movers. Patching their clean output into a corrupted run worsens the answer, proving they actively push toward the wrong name.

Positive name movers like L9H9 (~+0.25) and adjacent row-9 cells also appear, though with smaller magnitudes. They attend from the final token to the indirect object, copying that correct name directly to the logits.

The L8H6 / L8H10 / L7H9 band (~+0.28) functions as the S-inhibition heads. They carry the critical "this is the repeated subject" signal forward to the final token.

Earlier in the network, L5H5 (+0.32) and the fainter L3H0 operate as early duplicate/induction heads. Their specific role is to initially spot that repeated name to begin with.

This single heatmap illustrates the full circuit: detect the duplicate, move it to the end, and write the right name out.